# EE4685: Assignment 2  
**Student Names:** Adam El Haddouchi (5476526) & Naufal El Khatibi (5315778)  
**Date:** March 2026  

## Introduction
In this notebook, we study **credit card fraud detection** using machine learning. The goal is to classify fraudulent credit card transactions while dealing with the strong class imbalance present in the dataset.

We compare **non-Bayesian** and **Bayesian** approaches and discuss their strengths and limitations, as required by the assignment. Beyond raw predictive performance, we also consider uncertainty quantification and how it can inform operational decisions (e.g., routing uncertain predictions to a human reviewer).

## Dataset
The dataset was created by the Machine Learning Group at Universite Libre de Bruxelles (Dal Pozzolo, 2015) and contains credit card transactions made by European cardholders in **September 2013** over a period of **two days**. It consists of **284,807 transactions**, of which only **492 are fraudulent**. The positive class represents only **0.172%** of the data, making this a highly imbalanced classification problem.

Most input features are anonymized due to confidentiality constraints. The variables **V1** to **V28** are the result of a **PCA transformation** applied to the original transaction features. The only variables that were not transformed are:

- **Time**: the number of seconds elapsed between each transaction and the first transaction in the dataset  
- **Amount**: the transaction amount  
- **Class**: the target variable, where `1` indicates fraud and `0` indicates a normal transaction  

**Source:** Dal Pozzolo, A. (2015). *Adaptive Machine Learning for Credit Card Fraud Detection.* PhD Thesis, ULB. Dataset available on [Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud).

Because of the extreme class imbalance, standard accuracy is not useful here. A classifier that predicts "normal" for everything already reaches 99.83%. We therefore focus on metrics better suited to imbalanced problems, primarily **AUPRC** (Area Under the Precision-Recall Curve), alongside precision, recall, F1-score, ROC-AUC, and confusion matrices.

## Notebook Overview

1. **Understanding the data** - We inspect the dataset, class balance, and feature structure.
2. **Exploratory data analysis** - We look at feature distributions and check for data quality issues.
3. **Preprocessing and setup** - We prepare the data for modeling and define the experimental setup to avoid leakage.
4. **Modeling** - We implement and compare supervised and anomaly-detection methods, using both Bayesian and non-Bayesian formulations.
5. **Evaluation and discussion** - We evaluate using metrics suited to imbalanced data and discuss trade-offs between the approaches.

## 1. Dataset Loading and Exploration

We start by loading the dataset and inspecting its structure: shape, data types, missing values, summary statistics, and class distribution.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
np.random.seed(42)

# plot styling
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.dpi": 120,
})
sns.set_style("whitegrid")

# load data
df = pd.read_csv("../data/creditcard.csv")
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# basic info: shape, types, missing values
print(f"Shape: {df.shape}")
print(f"\nData types:\n{df.dtypes.value_counts()}")
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"\n--- Summary statistics ---")
df.describe()

In [ ]:
# class distribution - fraud is class 1
class_counts = df["Class"].value_counts().sort_index()
fraud_rate = class_counts[1] / len(df) * 100

print("Class distribution:")
print(f"  Normal (0): {class_counts[0]:,}")
print(f"  Fraud  (1): {class_counts[1]:,}")
print(f"  Fraud rate:  {fraud_rate:.4f}%")
print(f"  Imbalance ratio: 1 : {class_counts[0] // class_counts[1]}")

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["Normal (0)", "Fraud (1)"], class_counts.values, color=["#4C72B0", "#C44E52"])
ax.set_ylabel("Number of transactions")
ax.set_title("Class Distribution")
for bar, val in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2000,
            f"{val:,}", ha="center", va="bottom", fontsize=10)
ax.set_yscale("log")  # log scale since normal dwarfs fraud
plt.tight_layout()
plt.savefig("../figures/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

The dataset contains 284,807 transactions across 31 columns, all stored as `float64` except for the integer target variable `Class`. There are no missing values.

The V1 through V28 features come from PCA applied by the dataset creators. Their means are effectively zero (order of $10^{-15}$) and their standard deviations range from about 1.96 (V1) down to 0.33 (V28), so they are centered but not uniformly scaled. This spread of roughly 6x in standard deviation will matter for distance-based models later. `Amount` has a mean of 88.35 with a standard deviation of 250.12 (range 0 to 25,691), and `Time` ranges from 0 to 172,792 seconds (about two days). These two features are on very different scales from the V-features and will need standardization during preprocessing.

The class distribution confirms the extreme imbalance: 284,315 normal transactions versus only 492 fraudulent ones, a fraud rate of 0.1727% and an imbalance ratio of roughly 577:1. A trivial "predict all normal" classifier would achieve 99.83% accuracy, which is why we use AUPRC as our primary evaluation metric instead.

## 2. Exploratory Data Analysis

Before modeling, we explore the distributions of key features and how they differ between legitimate and fraudulent transactions. This helps inform preprocessing choices and gives a sense of which features might be useful for classification.

In [ ]:
# distribution of transaction Amount
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["Amount"], bins=100, color="#4C72B0", edgecolor="white", linewidth=0.3)
axes[0].set_title("Distribution of Transaction Amount")
axes[0].set_xlabel("Amount (€)")
axes[0].set_ylabel("Frequency")

# log scale to see the tail better
axes[1].hist(df["Amount"], bins=100, color="#4C72B0", edgecolor="white", linewidth=0.3)
axes[1].set_title("Distribution of Transaction Amount (log scale)")
axes[1].set_xlabel("Amount (€)")
axes[1].set_ylabel("Frequency (log)")
axes[1].set_yscale("log")

plt.tight_layout()
plt.savefig("../figures/amount_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Amount statistics:")
print(f"  Median: €{df['Amount'].median():.2f}")
print(f"  Mean:   €{df['Amount'].mean():.2f}")
print(f"  Max:    €{df['Amount'].max():.2f}")
print(f"  Skewness: {df['Amount'].skew():.2f}")
print(f"  % zero: {(df['Amount'] == 0).mean() * 100:.2f}%")

The distribution of transaction amounts is heavily right-skewed (skewness ~17). The median is only €22.00 while the mean is €88.35, reflecting the long right tail that extends up to €25,691. Most transactions are concentrated at low values, and 0.64% are exactly zero. The log-scale histogram shows that transactions span several orders of magnitude. A `log1p` transform (log because of the skew, the "+1" because of the zeros) followed by standardization seems appropriate for preprocessing.

In [ ]:
# distribution of Time (seconds since first transaction)
fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(df["Time"], bins=100, color="#4C72B0", edgecolor="white", linewidth=0.3)
ax.set_title("Distribution of Time (seconds since first transaction)")
ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Frequency")

# mark the approximate day boundary
seconds_per_day = 86400
ax.axvline(x=seconds_per_day, color="#C44E52", linestyle="--", linewidth=1.2, label="~24 hours")
ax.legend()

plt.tight_layout()
plt.savefig("../figures/time_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Time statistics:")
print(f"  Min:  {df['Time'].min():.0f}s")
print(f"  Max:  {df['Time'].max():.0f}s  ({df['Time'].max()/3600:.1f} hours)")
print(f"  Span: {df['Time'].max()/86400:.2f} days")

The `Time` feature spans about 172,792 seconds (~2 days), consistent with the dataset documentation. The histogram shows a clear daily cycle: transaction volume drops during nighttime hours and peaks during the day. This pattern repeats across both days. Like `Amount`, `Time` is not standardized and will need scaling.

In [ ]:
# split by class for comparison
fraud = df[df["Class"] == 1]
normal = df[df["Class"] == 0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Amount by class (density so we can compare despite different counts)
axes[0].hist(normal["Amount"], bins=80, alpha=0.7, label=f"Normal (n={len(normal):,})",
             color="#4C72B0", density=True, edgecolor="white", linewidth=0.3)
axes[0].hist(fraud["Amount"], bins=80, alpha=0.7, label=f"Fraud (n={len(fraud)})",
             color="#C44E52", density=True, edgecolor="white", linewidth=0.3)
axes[0].set_title("Amount Distribution by Class")
axes[0].set_xlabel("Amount (€)")
axes[0].set_ylabel("Density")
axes[0].set_xlim(0, 2000)  # cap at 2000 to see the bulk of the data
axes[0].legend()

# Time by class
axes[1].hist(normal["Time"], bins=80, alpha=0.7, label="Normal",
             color="#4C72B0", density=True, edgecolor="white", linewidth=0.3)
axes[1].hist(fraud["Time"], bins=80, alpha=0.7, label="Fraud",
             color="#C44E52", density=True, edgecolor="white", linewidth=0.3)
axes[1].set_title("Time Distribution by Class")
axes[1].set_xlabel("Time (seconds)")
axes[1].set_ylabel("Density")
axes[1].legend()

plt.tight_layout()
plt.savefig("../figures/amount_time_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Amount - Normal: mean=€{normal['Amount'].mean():.2f}, median=€{normal['Amount'].median():.2f}")
print(f"Amount - Fraud:  mean=€{fraud['Amount'].mean():.2f}, median=€{fraud['Amount'].median():.2f}")
print(f"\nTime - Normal: mean={normal['Time'].mean()/3600:.1f}h")
print(f"Time - Fraud:  mean={fraud['Time'].mean()/3600:.1f}h")

There are some differences between classes, especially for `Amount`. Fraudulent transactions have a higher mean (€122.21 vs. €88.29) but a much lower median (€9.25 vs. €22.00). So most fraudulent transactions are small, but a subset involves larger sums that pull the mean up. This gap between median and mean suggests `Amount` carries some nonlinear discriminative signal even though its linear correlation with `Class` is near zero.

For `Time`, the distributions look broadly similar between classes (mean ~22.4h for fraud vs. ~26.3h for normal). Both follow the same daily cycle. There is no strong evidence that fraud happens preferentially at certain times, so `Time` is likely less useful for classification.

In [ ]:
# correlation of each feature with the fraud label
v_features = [f"V{i}" for i in range(1, 29)]
all_features = v_features + ["Amount", "Time"]

correlations = df[all_features + ["Class"]].corr()["Class"].drop("Class").sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#C44E52" if v < 0 else "#4C72B0" for v in correlations.values]
correlations.plot(kind="barh", ax=ax, color=colors)
ax.set_title("Pearson Correlation of Features with Class (Fraud)")
ax.set_xlabel("Correlation coefficient")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.savefig("../figures/feature_correlation_with_class.png", dpi=150, bbox_inches="tight")
plt.show()

# top features by absolute correlation
print("Top 10 features most correlated with fraud (by absolute value):")
top_corr = correlations.abs().sort_values(ascending=False).head(10)
for feat, val in top_corr.items():
    sign = "+" if correlations[feat] > 0 else "-"
    print(f"  {feat:>8s}: {sign}{val:.4f}")

Most V-features have weak individual correlations with the fraud label, but a few stand out. The strongest negative correlations (lower values associated with fraud) are V17 (r = -0.33), V14 (r = -0.30), V12 (r = -0.26), V10 (r = -0.22), V16 (r = -0.20), V3 (r = -0.19), and V7 (r = -0.19). On the positive side, V11 (+0.15) and V4 (+0.13) show the strongest associations.

Since V1-V28 are PCA components of unknown original features, we cannot assign any semantic meaning to them. We can only observe that some components carry more discriminative signal than others. `Amount` and `Time` both show negligible linear correlation with `Class` (below 0.01), though the class-conditional analysis above suggested `Amount` may still carry nonlinear information.

The box plots below show the top correlated V-features split by class.

In [ ]:
# box plots of the V-features most correlated with fraud
top_features = ["V17", "V14", "V12", "V10", "V16", "V3", "V7", "V11"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    data = [normal[feat].values, fraud[feat].values]
    bp = axes[i].boxplot(data, labels=["Normal", "Fraud"], patch_artist=True,
                         widths=0.6, showfliers=False,
                         medianprops=dict(color="black", linewidth=1.5))
    bp["boxes"][0].set_facecolor("#4C72B0")
    bp["boxes"][0].set_alpha(0.7)
    bp["boxes"][1].set_facecolor("#C44E52")
    bp["boxes"][1].set_alpha(0.7)
    axes[i].set_title(f"{feat}  (r = {correlations[feat]:.3f})")
    axes[i].set_ylabel("Value")

plt.suptitle("Top Correlated V-Features: Distribution by Class", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("../figures/top_v_features_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

The box plots confirm clear distributional shifts between classes for these features. For the negatively correlated features, fraud transactions are concentrated at much lower values than normal ones, with V14 and V12 showing the most pronounced separation. V11 (positively correlated) shows the opposite pattern: fraud transactions tend to have higher values. In all eight features, the interquartile ranges of the two classes have limited overlap, so these components carry meaningful discriminative information even individually.

We cannot say what these PCA components represent in terms of the original transaction features. We can only note that they encode patterns that differ systematically between legitimate and fraudulent activity.

In [ ]:
# check for constant or near-constant features
print("Unique values per feature:")
nunique = df[all_features].nunique().sort_values()
print(nunique.to_string())
print(f"\nMin unique values: {nunique.min()} ({nunique.idxmin()})")
print(f"Any constant features (nunique == 1): {(nunique == 1).any()}")

# also check variance as a sanity check
low_var = df[all_features].var().sort_values()
print(f"\nLowest-variance features:")
for feat, var in low_var.head(5).items():
    print(f"  {feat}: var = {var:.4f}")

No features are constant or near-constant. All 30 features have at least tens of thousands of unique values, so there is no reason to drop any feature before modeling. The lowest-variance features are the higher-numbered V-components (V28, V27, V26), which is expected since PCA components are ordered by explained variance.

### EDA Summary

Key takeaways from the exploratory analysis:

- **Class imbalance** is extreme (0.17% fraud). Accuracy is not a useful metric; we will use AUPRC.
- **Amount** is heavily right-skewed and needs a `log1p` transform before standardization. It shows nonlinear discriminative signal (fraud median €9.25 vs. normal median €22.00) despite near-zero linear correlation with Class.
- **Time** shows daily cyclical patterns but similar distributions across classes. Low discriminative value.
- **V-features** are centered (from PCA) but not uniformly scaled (std ranges from 0.33 to 1.96). All features should be standardized, especially for distance-based models like OC-SVM and BGMM.
- **Top discriminative features**: V17 (r = -0.33), V14 (r = -0.30), V12 (r = -0.26), V10 (r = -0.22).
- **No constant or near-constant features** exist; all 30 features are retained.

## 3. Preprocessing and Train/Validation/Test Split

The EDA revealed two preprocessing needs. First, `Amount` is heavily right-skewed (skewness ~17) and contains zero values, so we apply a `log1p` transform to compress the tail before standardization. Second, the V-features have standard deviations ranging from 0.33 (V28) to 1.96 (V1), a 6x spread that would bias distance-based models. We standardize all 30 features (V1-V28, log-transformed Amount, and Time) using `StandardScaler`.

To prevent data leakage, we follow a strict ordering: (1) apply `log1p` to Amount on the full DataFrame, since this is a fixed mathematical transform that does not depend on data statistics; (2) perform a stratified train/validation/test split; (3) fit the scaler on the training set only, then transform all three splits.

For handling the class imbalance, we use `class_weight='balanced'` in supervised models rather than SMOTE. SMOTE generates synthetic minority samples by interpolating between existing ones, but in a 30-dimensional PCA-transformed space where the original feature semantics are unknown, the validity of those interpolated points is questionable. Oversampling also inflates the training set and increases computation time. Adjusting class weights penalizes misclassification of the minority class more heavily without introducing synthetic data points.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# log1p transform on Amount (data-independent, safe to do before splitting)
df["LogAmount"] = np.log1p(df["Amount"])

print(f"Amount  -> skewness: {df['Amount'].skew():.2f}, mean: {df['Amount'].mean():.2f}, std: {df['Amount'].std():.2f}")
print(f"LogAmount -> skewness: {df['LogAmount'].skew():.2f}, mean: {df['LogAmount'].mean():.2f}, std: {df['LogAmount'].std():.2f}")

# replace Amount with LogAmount in the feature list, keep Time
features = [f"V{i}" for i in range(1, 29)] + ["LogAmount", "Time"]
print(f"\nFeature set ({len(features)} features): {features[:3]} ... {features[-3:]}")

X = df[features].values
y = df["Class"].values

# stratified split: 80/20 -> train+val / test, then 80/20 -> train / val
# gives us 64/16/20 overall
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.20, stratify=y_temp, random_state=42
)

print(f"\nSplit sizes:")
print(f"  Train:      {X_train.shape[0]:>7,} samples  ({X_train.shape[0]/len(df)*100:.1f}%),  fraud: {y_train.sum():>4}  ({y_train.mean()*100:.3f}%)")
print(f"  Validation: {X_val.shape[0]:>7,} samples  ({X_val.shape[0]/len(df)*100:.1f}%),  fraud: {y_val.sum():>4}  ({y_val.mean()*100:.3f}%)")
print(f"  Test:       {X_test.shape[0]:>7,} samples  ({X_test.shape[0]/len(df)*100:.1f}%),  fraud: {y_test.sum():>4}  ({y_test.mean()*100:.3f}%)")
print(f"  Total:      {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]:>7,}")
print(f"\nFraud rate consistency check:")
print(f"  Overall: {y.mean()*100:.4f}%")
print(f"  Train:   {y_train.mean()*100:.4f}%")
print(f"  Val:     {y_val.mean()*100:.4f}%")
print(f"  Test:    {y_test.mean()*100:.4f}%")

The `log1p` transform reduced the skewness of `Amount` from 16.98 to 0.16, making it roughly symmetric. The standard deviation dropped from 250.12 to 1.66, which is now comparable to the V-features.

The stratified split produced 182,276 training samples (64.0%), 45,569 validation samples (16.0%), and 56,962 test samples (20.0%), totaling 284,807. The fraud rate is consistent across all three splits: 0.173% in train (315 cases), 0.173% in validation (79 cases), and 0.172% in test (98 cases). The stratification preserved the class distribution as intended.

In [ ]:
# fit scaler on training set only, then transform all splits
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)     # transform only, no fitting
X_test = scaler.transform(X_test)   # transform only, no fitting

# sanity check: training set should have mean ~0 and std ~1
train_means = X_train.mean(axis=0)
train_stds = X_train.std(axis=0)

print("Sanity check - training set statistics after scaling:")
print(f"  Mean range: [{train_means.min():.6f}, {train_means.max():.6f}]  (should be ~0)")
print(f"  Std  range: [{train_stds.min():.6f}, {train_stds.max():.6f}]  (should be ~1)")

# spot-check a few features
print(f"\nPer-feature check (first 5 + LogAmount + Time):")
check_idx = [0, 1, 2, 3, 4, 28, 29]  # V1-V5, LogAmount, Time
for i in check_idx:
    print(f"  {features[i]:>10s}: mean = {train_means[i]:+.6f}, std = {train_stds[i]:.6f}")

# normal-only training set for anomaly detection models
X_train_normal = X_train[y_train == 0]
print(f"\nAnomaly detection training set (normal only): {X_train_normal.shape[0]:,} samples")
print(f"  Fraud excluded: {y_train.sum()} samples")
print(f"\nFinal array shapes:")
print(f"  X_train:        {X_train.shape}")
print(f"  X_val:          {X_val.shape}")
print(f"  X_test:         {X_test.shape}")
print(f"  X_train_normal: {X_train_normal.shape}")

After scaling, all 30 features have mean 0 and standard deviation 1 on the training set, confirming the scaler was applied correctly. The scaler was fit on the training set only and then used to transform the validation and test sets, so there is no information leakage.

For the anomaly detection models (OC-SVM, BGMM, Autoencoder), we created a separate training set containing only the 181,961 normal transactions from the training split. These models learn the distribution of normal behavior and flag deviations as potential fraud. The 315 fraudulent training samples are excluded from this set but remain available in the full training set for supervised models.

The preprocessing pipeline is now complete. To summarize the setup going forward:
- **Supervised models** (Logistic Regression, Bayesian Logistic Regression): train on `X_train`, `y_train` with `class_weight='balanced'`, tune on `X_val`, `y_val`, evaluate on `X_test`, `y_test`.
- **Anomaly detection models** (OC-SVM, BGMM, Autoencoder): train on `X_train_normal`, tune thresholds on `X_val`, `y_val`, evaluate on `X_test`, `y_test`.

## 4. Evaluation Framework

Before building any models, we establish the evaluation criteria. The extreme class imbalance in this dataset (0.17% fraud) makes standard metrics unreliable, and different model families require different evaluation considerations. This section lays out our metric choices and their justification.

### Why AUPRC is the primary metric

The Area Under the Precision-Recall Curve (AUPRC) is our primary evaluation metric. This choice is motivated by the class imbalance: with only 492 fraudulent transactions out of 284,807, metrics that depend on the large number of true negatives give an inflated picture of model quality.

Davis and Goadrich (2006) showed that precision-recall curves provide a more informative picture than ROC curves when datasets are heavily skewed, because precision is directly sensitive to false positives without dilution from the large negative class. ROC curves use the false positive rate, FPR = FP / (FP + TN), as the x-axis. When TN is enormous (here ~284,000), even hundreds of false positives barely move the FPR, making the ROC curve look deceptively good. Precision, on the other hand, is TP / (TP + FP), which exposes false positives directly.

Saito and Rehmsmeier (2015) provided further empirical evidence that ROC plots can be visually deceptive for imbalanced datasets, and recommended precision-recall plots as the default for such problems. Their study demonstrated that classifiers with near-identical ROC curves can have very different precision-recall profiles, meaning ROC-AUC can fail to distinguish between models that differ substantially in practical performance.

We still report ROC-AUC as a secondary metric for completeness, but all model comparisons and conclusions are based on AUPRC.

### Why accuracy is useless here

To make the case concrete, consider the simplest possible classifier: one that predicts "normal" for every transaction, regardless of input.

In [ ]:
# trivial classifier: predict "normal" for everything
n_total = len(y)
n_normal = (y == 0).sum()
trivial_accuracy = n_normal / n_total

print(f"Trivial 'predict all normal' classifier:")
print(f"  Correct predictions: {n_normal:,} out of {n_total:,}")
print(f"  Accuracy: {trivial_accuracy * 100:.2f}%")
print(f"  Frauds detected: 0 out of {(y == 1).sum()}")
print(f"  Recall: 0.00%")

A classifier that blindly labels every transaction as legitimate achieves 99.83% accuracy. It is, of course, completely useless: it catches zero fraud. This demonstrates why accuracy is not a meaningful metric for this problem. Any model we build must be evaluated on its ability to identify the rare positive class, not on its ability to correctly label the overwhelming majority of normal transactions.

### Evaluation protocol for supervised vs anomaly detection models

Our model lineup includes two types of learners that differ in what training data they see. Supervised models (Logistic Regression and Bayesian Logistic Regression) are trained on both normal and fraudulent transactions, with access to the class labels. Anomaly detection models (One-Class SVM, Bayesian Gaussian Mixture Model, and Autoencoder) are trained on normal transactions only and have never seen a fraudulent example during training. This information asymmetry means that comparing the two groups is not an apples-to-apples exercise.

To keep the comparison fair where possible, all models are evaluated on the same held-out test set using the same metrics (AUPRC, ROC-AUC, precision, recall, F1, and confusion matrices). The test set was not used during training or threshold tuning for any model. Thresholds are tuned on the validation set for all models (Phase 9). The key interpretive point is that supervised models have a structural advantage: they were told what fraud looks like. If an anomaly detector approaches supervised performance despite this handicap, that is a strong result. If it falls short, the gap quantifies how much labeled fraud data is worth in this setting.

## 5. Supervised Models

### 5.1 Logistic Regression (Non-Bayesian Baseline)

Logistic Regression is our starting point for binary classification. It models the log-odds of the positive class as a linear function of the input features and outputs calibrated probabilities through the sigmoid function. In fraud detection, LR is one of the most commonly deployed models in production (Dal Pozzolo et al., 2014), partly because it is interpretable and computationally cheap.

In our project, LR is the non-Bayesian supervised baseline. All other models will be compared against it, and if a more complex model does not clearly beat LR, that is itself a useful finding. We use `class_weight='balanced'` to handle the class imbalance. This scales the loss contribution of each class inversely proportional to its frequency, giving the 315 fraud samples in the training set roughly 577 times more weight per sample than the normal transactions.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    roc_curve, roc_auc_score,
    confusion_matrix, classification_report, f1_score
)

# train logistic regression with balanced class weights
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    solver='lbfgs'
)
lr_model.fit(X_train, y_train)

# predict on validation set
lr_val_probs = lr_model.predict_proba(X_val)[:, 1]  # probability of fraud
lr_val_preds = lr_model.predict(X_val)               # binary predictions at default 0.5 threshold

print(f"Logistic Regression trained on {X_train.shape[0]:,} samples ({y_train.sum()} fraud)")
print(f"Validation set: {X_val.shape[0]:,} samples ({y_val.sum()} fraud)")
print(f"\nCoefficients shape: {lr_model.coef_.shape}")
print(f"Intercept: {lr_model.intercept_[0]:.4f}")
print(f"Number of iterations: {lr_model.n_iter_[0]}")

In [ ]:
# PR curve and ROC curve for LR on the validation set
lr_precision, lr_recall, lr_pr_thresholds = precision_recall_curve(y_val, lr_val_probs)
lr_auprc = average_precision_score(y_val, lr_val_probs)

lr_fpr, lr_tpr, lr_roc_thresholds = roc_curve(y_val, lr_val_probs)
lr_roc_auc = roc_auc_score(y_val, lr_val_probs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR curve - more informative for imbalanced data
axes[0].plot(lr_recall, lr_precision, color="#4C72B0", linewidth=2, label=f"LR (AUPRC = {lr_auprc:.4f})")
axes[0].axhline(y=y_val.mean(), color="gray", linestyle="--", linewidth=1, label=f"Baseline (prevalence = {y_val.mean():.4f})")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall Curve (Validation Set)")
axes[0].legend(loc="upper right")
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.05])

# ROC curve - will look good due to large TN count
axes[1].plot(lr_fpr, lr_tpr, color="#4C72B0", linewidth=2, label=f"LR (ROC-AUC = {lr_roc_auc:.4f})")
axes[1].plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1, label="Random classifier")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve (Validation Set)")
axes[1].legend(loc="lower right")
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig("../figures/lr_pr_roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"AUPRC:   {lr_auprc:.4f}")
print(f"ROC-AUC: {lr_roc_auc:.4f}")

In [ ]:
# confusion matrix and classification metrics at default 0.5 threshold
lr_cm = confusion_matrix(y_val, lr_val_preds)
lr_f1 = f1_score(y_val, lr_val_preds)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(lr_cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Normal", "Fraud"], yticklabels=["Normal", "Fraud"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix - Logistic Regression (threshold = 0.5)")
plt.tight_layout()
plt.savefig("../figures/confusion_matrix_lr.png", dpi=150, bbox_inches="tight")
plt.show()

tn, fp, fn, tp = lr_cm.ravel()
print(f"Confusion matrix (threshold = 0.5):")
print(f"  True Negatives:  {tn:>6,}")
print(f"  False Positives: {fp:>6,}")
print(f"  False Negatives: {fn:>6,}")
print(f"  True Positives:  {tp:>6,}")
print(f"\nPrecision: {tp/(tp+fp):.4f}")
print(f"Recall:    {tp/(tp+fn):.4f}")
print(f"F1 Score:  {lr_f1:.4f}")
print(f"\nNote: these metrics use the default 0.5 threshold.")
print(f"Threshold tuning on the validation set will be done in Phase 9.")

The Logistic Regression model converged in 21 iterations. On the validation set, it achieves an AUPRC of 0.6825 and a ROC-AUC of 0.9729. The gap between these two numbers is telling: ROC-AUC looks high because the false positive rate denominator includes all ~45,490 actual negatives, which dilutes even a large number of false positives. AUPRC uses precision instead of FPR, giving a less optimistic but more practically useful picture.

At the default 0.5 classification threshold, the model catches 69 out of 79 fraudulent transactions (recall = 0.87) but generates 1,121 false positives, giving a precision of just 5.8%. That means roughly 94% of fraud alerts at this threshold are false alarms. The F1 score is 0.109, which reflects this extreme precision-recall tradeoff. The `class_weight='balanced'` setting pushed the model toward high recall, but at the cost of many false positives. Using `class_weight='balanced'` also slightly affects the learned coefficients compared to an unweighted model and may modestly reduce AUPRC, but we use it for consistency with the plan's imbalance handling strategy across all supervised models.

These numbers use the default 0.5 threshold, which is not optimal for a problem with a 0.17% base rate. Threshold tuning on the validation set in Phase 9 should improve the F1 by finding a better operating point on the precision-recall curve. For now, the AUPRC of 0.6825 is the threshold-independent summary of LR's performance and serves as our baseline for all subsequent models.

### 5.2 Bayesian Logistic Regression

Logistic Regression produces a single point estimate for each weight parameter by maximizing the (penalized) log-likelihood. Bayesian Logistic Regression treats the weight vector as a random variable with a prior distribution and computes a posterior distribution over weights given the training data. Instead of a single predicted probability for each transaction, the model yields a distribution of predicted probabilities, which lets us quantify how uncertain the model is about each prediction.

The posterior over the weights is intractable for logistic regression because the logistic likelihood does not conjugate with a Gaussian prior (Bishop, 2006, Ch. 4). We therefore use the Laplace approximation: we find the mode of the posterior (which coincides with the MAP estimate, equivalent to L2-regularized logistic regression) and fit a Gaussian centered at that mode, with covariance equal to the inverse of the Hessian of the negative log-posterior evaluated at the mode. This gives us a tractable Gaussian approximation to the posterior: $p(\mathbf{w} | \mathcal{D}) \approx \mathcal{N}(\mathbf{w}_\text{MAP}, \mathbf{H}^{-1})$, where $\mathbf{H}$ is the Hessian of the negative log-posterior.

To generate predictions with uncertainty, we propagate the posterior weight distribution through the logistic function. For each test point $\mathbf{x}$, the logit $a = \mathbf{w}^\top \mathbf{x}$ is approximately Gaussian with mean $\mu_a = \mathbf{w}_\text{MAP}^\top \mathbf{x}$ and variance $\sigma_a^2 = \mathbf{x}^\top \mathbf{H}^{-1} \mathbf{x}$. We then use the probit approximation to integrate the logistic sigmoid over this Gaussian, yielding a predictive probability that accounts for parameter uncertainty (Bishop, 2006, Section 4.5.2).

In [ ]:
from scipy.special import expit  # sigmoid function
from scipy.linalg import cho_factor, cho_solve

# Bayesian LR via Laplace approximation
# use the already-trained LR weights as the MAP estimate
# (scikit-learn's L2 penalty = Gaussian prior, so MLE = MAP)
w_map = np.concatenate([lr_model.coef_.ravel(), lr_model.intercept_])  # (31,)
n_features = X_train.shape[1]

# add intercept column to design matrix
X_train_aug = np.hstack([X_train, np.ones((X_train.shape[0], 1))])  # (n_train, 31)
X_val_aug = np.hstack([X_val, np.ones((X_val.shape[0], 1))])

# Hessian of the negative log-posterior at w_MAP:
# H = X^T diag(pi*(1-pi)) X + lambda * I
# pi = sigmoid(X @ w_MAP), lambda = regularization strength
lambda_reg = 1.0 / lr_model.C  # C = 1/lambda, default C=1.0

# predicted probabilities at MAP
pi_train = expit(X_train_aug @ w_map)

# weight matrix: pi * (1 - pi) for each training sample
S = pi_train * (1 - pi_train)  # (n_train,)

# account for class_weight='balanced' in the Hessian
n_samples = len(y_train)
n_pos = y_train.sum()
n_neg = n_samples - n_pos
w_pos = n_samples / (2.0 * n_pos)   # weight for fraud class
w_neg = n_samples / (2.0 * n_neg)   # weight for normal class
sample_weights = np.where(y_train == 1, w_pos, w_neg)

# class-weighted Hessian: H = X^T @ diag(sample_weights * S) @ X + lambda*I
SW = S * sample_weights  # (n_train,)
H = (X_train_aug.T * SW) @ X_train_aug  # (31, 31)

# add prior precision (L2 reg), but don't regularize the intercept
prior_precision = np.eye(len(w_map)) * lambda_reg
prior_precision[-1, -1] = 0
H += prior_precision

# posterior covariance = inverse Hessian (Cholesky for stability)
try:
    L, low = cho_factor(H)
    H_inv = cho_solve((L, low), np.eye(len(w_map)))
    print("Hessian inverted successfully via Cholesky decomposition.")
except np.linalg.LinAlgError:
    H_inv = np.linalg.pinv(H)
    print("Warning: Hessian near-singular, used pseudo-inverse.")

# predictive distribution on the validation set
# logit mean = w_MAP^T x, logit variance = x^T H^{-1} x
blr_logit_mean = X_val_aug @ w_map
blr_logit_var = np.sum(X_val_aug @ H_inv * X_val_aug, axis=1)
blr_logit_std = np.sqrt(np.maximum(blr_logit_var, 0))  # clip negative numerical noise

# probit approximation: integrate sigmoid over Gaussian uncertainty in the logit
kappa = 1.0 / np.sqrt(1.0 + np.pi * blr_logit_var / 8.0)
blr_val_probs = expit(kappa * blr_logit_mean)

print(f"\nBayesian LR (Laplace approximation):")
print(f"  MAP weights shape: {w_map.shape}")
print(f"  Posterior covariance shape: {H_inv.shape}")
print(f"  Posterior weight std (diagonal): min={np.sqrt(np.diag(H_inv)).min():.6f}, max={np.sqrt(np.diag(H_inv)).max():.6f}")
print(f"\nValidation predictions:")
print(f"  Logit mean range: [{blr_logit_mean.min():.2f}, {blr_logit_mean.max():.2f}]")
print(f"  Logit std range:  [{blr_logit_std.min():.4f}, {blr_logit_std.max():.4f}]")
print(f"  Predictive prob range: [{blr_val_probs.min():.6f}, {blr_val_probs.max():.6f}]")

The Laplace approximation succeeded: the Hessian was positive definite and inverted cleanly via Cholesky decomposition. The posterior standard deviations on individual weights are small (ranging from about 0.011 to 0.039), reflecting the large training set of 182,276 samples. With this much data, the posterior is tightly concentrated around the MAP estimate.

The logit-space uncertainty varies across validation samples, with standard deviations between 0.03 and 2.60. Samples with higher logit variance are those where the model is less certain about its prediction. The predictive probabilities (after integrating out the weight uncertainty via the probit approximation) range from near zero to near one, as expected for a classification task.

The key difference from standard LR is that we now have a measure of epistemic uncertainty for each prediction. In the next cells, we evaluate predictive performance and then use this uncertainty to build a decision protocol.

In [ ]:
# PR curve and ROC curve for Bayesian LR on validation set
blr_precision_curve, blr_recall_curve, blr_pr_thresholds = precision_recall_curve(y_val, blr_val_probs)
blr_auprc = average_precision_score(y_val, blr_val_probs)

blr_fpr, blr_tpr, blr_roc_thresholds = roc_curve(y_val, blr_val_probs)
blr_roc_auc = roc_auc_score(y_val, blr_val_probs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR curves: BLR vs LR
axes[0].plot(blr_recall_curve, blr_precision_curve, color="#C44E52", linewidth=2,
             label=f"Bayesian LR (AUPRC = {blr_auprc:.4f})")
axes[0].plot(lr_recall, lr_precision, color="#4C72B0", linewidth=2, alpha=0.6,
             label=f"LR (AUPRC = {lr_auprc:.4f})")
axes[0].axhline(y=y_val.mean(), color="gray", linestyle="--", linewidth=1,
                label=f"Baseline (prevalence = {y_val.mean():.4f})")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall Curve (Validation Set)")
axes[0].legend(loc="upper right")
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.05])

# ROC curves: BLR vs LR - will look very similar
axes[1].plot(blr_fpr, blr_tpr, color="#C44E52", linewidth=2,
             label=f"Bayesian LR (ROC-AUC = {blr_roc_auc:.4f})")
axes[1].plot(lr_fpr, lr_tpr, color="#4C72B0", linewidth=2, alpha=0.6,
             label=f"LR (ROC-AUC = {lr_roc_auc:.4f})")
axes[1].plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1, label="Random")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve (Validation Set)")
axes[1].legend(loc="lower right")
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig("../figures/blr_pr_roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# confusion matrix at 0.5 threshold
blr_val_preds = (blr_val_probs >= 0.5).astype(int)
blr_cm = confusion_matrix(y_val, blr_val_preds)
blr_f1 = f1_score(y_val, blr_val_preds)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(blr_cm, annot=True, fmt="d", cmap="Reds", ax=ax,
            xticklabels=["Normal", "Fraud"], yticklabels=["Normal", "Fraud"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix - Bayesian LR (threshold = 0.5)")
plt.tight_layout()
plt.savefig("../figures/confusion_matrix_blr.png", dpi=150, bbox_inches="tight")
plt.show()

tn, fp, fn, tp = blr_cm.ravel()
print(f"Bayesian LR vs Logistic Regression (validation set):")
print(f"{'Metric':<15s} {'BLR':>10s} {'LR':>10s}")
print(f"{'-'*37}")
print(f"{'AUPRC':<15s} {blr_auprc:>10.4f} {lr_auprc:>10.4f}")
print(f"{'ROC-AUC':<15s} {blr_roc_auc:>10.4f} {lr_roc_auc:>10.4f}")
print(f"{'F1 (t=0.5)':<15s} {blr_f1:>10.4f} {lr_f1:>10.4f}")
print(f"{'Precision':<15s} {tp/(tp+fp):>10.4f} {lr_cm.ravel()[3]/(lr_cm.ravel()[3]+lr_cm.ravel()[1]):>10.4f}")
print(f"{'Recall':<15s} {tp/(tp+fn):>10.4f} {lr_cm.ravel()[3]/(lr_cm.ravel()[3]+lr_cm.ravel()[2]):>10.4f}")
print(f"\nConfusion matrix (threshold = 0.5):")
print(f"  TP={tp}, FP={fp}, FN={fn}, TN={tn}")

The Bayesian LR achieves an AUPRC of 0.6878, compared to 0.6825 for standard LR. The ROC-AUC values are nearly identical (0.9730 vs 0.9729). At the default 0.5 threshold, the confusion matrices are the same: 69 true positives, 1,121 false positives, 10 false negatives, and 44,369 true negatives.

This outcome is consistent with what we expected. Both models share the same MAP weights (the Laplace approximation centers the posterior at the MAP estimate, which is the L2-regularized MLE that scikit-learn already found). The probit approximation modulates the predictive probabilities slightly by shrinking extreme predictions toward 0.5 in regions of higher uncertainty, which produces a marginal improvement in AUPRC. But the ranking of transactions is largely preserved, so the curves nearly overlap.

The real value of Bayesian LR is not in these aggregate metrics. It is in the per-prediction uncertainty estimates, which we use next to build a decision protocol.

In [ ]:
# Visualize uncertainty: predicted probability vs logit-space std
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter of predicted prob vs uncertainty, colored by true label
fraud_mask = y_val == 1
normal_mask = y_val == 0

axes[0].scatter(blr_val_probs[normal_mask], blr_logit_std[normal_mask],
                alpha=0.1, s=3, c="#4C72B0", label="Normal", rasterized=True)
axes[0].scatter(blr_val_probs[fraud_mask], blr_logit_std[fraud_mask],
                alpha=0.8, s=20, c="#C44E52", label="Fraud", zorder=5)
axes[0].set_xlabel("Predicted Fraud Probability")
axes[0].set_ylabel("Logit-Space Uncertainty (std)")
axes[0].set_title("Prediction vs Uncertainty (Bayesian LR)")
axes[0].legend()

# Right: distribution of uncertainty for fraud vs normal
axes[1].hist(blr_logit_std[normal_mask], bins=80, alpha=0.7, density=True,
             color="#4C72B0", label="Normal", edgecolor="white", linewidth=0.3)
axes[1].hist(blr_logit_std[fraud_mask], bins=30, alpha=0.7, density=True,
             color="#C44E52", label="Fraud", edgecolor="white", linewidth=0.3)
axes[1].set_xlabel("Logit-Space Uncertainty (std)")
axes[1].set_ylabel("Density")
axes[1].set_title("Uncertainty Distribution by Class")
axes[1].legend()

plt.tight_layout()
plt.savefig("../figures/blr_uncertainty_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

# summary stats
print("Logit-space uncertainty by class:")
print(f"  Normal: mean={blr_logit_std[normal_mask].mean():.4f}, "
      f"median={np.median(blr_logit_std[normal_mask]):.4f}, "
      f"95th pct={np.percentile(blr_logit_std[normal_mask], 95):.4f}")
print(f"  Fraud:  mean={blr_logit_std[fraud_mask].mean():.4f}, "
      f"median={np.median(blr_logit_std[fraud_mask]):.4f}, "
      f"95th pct={np.percentile(blr_logit_std[fraud_mask], 95):.4f}")

The uncertainty plot reveals a clear pattern. Normal transactions cluster at low predicted probability and low uncertainty (logit std median 0.073, 95th percentile 0.145). Fraud transactions have substantially higher uncertainty (logit std median 0.550, 95th percentile 1.900), roughly 8 times higher than normal transactions on average.

This is expected. The model was trained on 182,276 samples, of which only 315 were fraudulent. The posterior is well-informed about what normal transactions look like (many training examples, tight posterior in that region of feature space) but less certain about the fraud region (few training examples, wider posterior). This asymmetry in epistemic uncertainty is exactly the kind of information that a non-Bayesian model cannot provide.

The scatter plot also shows that fraud cases tend to appear at higher predicted probabilities with a wide spread in uncertainty, while the bulk of normal transactions occupy the low-probability, low-uncertainty corner. This separation is what motivates the three-bucket decision protocol in the next section.

### 5.3 Bayesian Decision Protocol

A standard fraud detection system makes binary decisions: flag or approve. With Bayesian uncertainty estimates, we can define a three-bucket decision protocol that routes transactions differently based on both the predicted probability and the model's confidence:

- **Auto-approve:** The model is confident the transaction is legitimate (low predicted probability, low uncertainty). No human intervention needed.
- **Auto-flag:** The model is confident the transaction is fraudulent (high predicted probability, low uncertainty). Block or flag automatically.
- **Human review:** The model is uncertain about its prediction (high uncertainty, regardless of predicted probability). Route to a human investigator.

This protocol has a practical advantage: it concentrates human effort on the cases where the model genuinely does not know the answer, rather than flooding investigators with confident false positives or letting uncertain frauds slip through. The thresholds that define the buckets are operational parameters that a fraud team would calibrate based on their capacity for manual review and their tolerance for missed fraud.

In [ ]:
# decision protocol: classify transactions into three buckets
# uncertainty threshold = 95th percentile of normal transaction uncertainty
uncertainty_threshold = np.percentile(blr_logit_std[normal_mask], 95)
prob_threshold = 0.5  # probability threshold for flag vs approve

print(f"Decision protocol thresholds:")
print(f"  Uncertainty threshold (logit std): {uncertainty_threshold:.4f}")
print(f"  Probability threshold: {prob_threshold}")
print(f"  (Uncertainty threshold = 95th percentile of normal transaction uncertainty)")

# assign each validation transaction to a bucket
bucket = np.full(len(y_val), "human_review", dtype=object)

# confident predictions (low uncertainty) get auto-flagged or auto-approved
confident = blr_logit_std <= uncertainty_threshold
bucket[confident & (blr_val_probs >= prob_threshold)] = "auto_flag"
bucket[confident & (blr_val_probs < prob_threshold)] = "auto_approve"
# everything else stays as human_review

# count transactions in each bucket
for b in ["auto_approve", "auto_flag", "human_review"]:
    mask = bucket == b
    n_total_b = mask.sum()
    n_fraud_b = y_val[mask].sum()
    n_normal_b = n_total_b - n_fraud_b
    pct = n_total_b / len(y_val) * 100
    print(f"\n{b:>15s}: {n_total_b:>6,} transactions ({pct:5.2f}%)")
    print(f"{'':>15s}  fraud: {n_fraud_b:>4}, normal: {n_normal_b:>6}")
    if b == "auto_flag" and n_total_b > 0:
        prec = n_fraud_b / n_total_b if n_total_b > 0 else 0
        recall = n_fraud_b / y_val.sum() if y_val.sum() > 0 else 0
        print(f"{'':>15s}  precision: {prec:.4f}, recall: {recall:.4f}")

# fraud accounting
print(f"\nTotal fraud in validation set: {y_val.sum()}")
fraud_in_review = y_val[bucket == "human_review"].sum()
fraud_in_approve = y_val[bucket == "auto_approve"].sum()
fraud_in_flag = y_val[bucket == "auto_flag"].sum()
print(f"Fraud caught by auto-flag: {fraud_in_flag}")
print(f"Fraud routed to human review: {fraud_in_review}")
print(f"Fraud missed (auto-approved): {fraud_in_approve}")

In [ ]:
# Visualize the decision protocol
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: scatter plot with bucket coloring
colors_map = {"auto_approve": "#2ca02c", "auto_flag": "#C44E52", "human_review": "#ff7f0e"}
labels_map = {"auto_approve": "Auto-approve", "auto_flag": "Auto-flag", "human_review": "Human review"}

for b in ["auto_approve", "human_review", "auto_flag"]:
    mask = bucket == b
    alpha = 0.05 if b == "auto_approve" else 0.3
    s = 3 if b == "auto_approve" else 15
    axes[0].scatter(blr_val_probs[mask], blr_logit_std[mask],
                    alpha=alpha, s=s, c=colors_map[b], label=labels_map[b], rasterized=True)

# overlay fraud cases with black edge
axes[0].scatter(blr_val_probs[fraud_mask], blr_logit_std[fraud_mask],
                s=30, facecolors='none', edgecolors='black', linewidth=1.2,
                label="Actual fraud", zorder=10)

# draw threshold lines
axes[0].axhline(y=uncertainty_threshold, color="gray", linestyle="--", linewidth=1,
                label=f"Uncertainty threshold = {uncertainty_threshold:.3f}")
axes[0].axvline(x=prob_threshold, color="gray", linestyle=":", linewidth=1,
                label=f"Prob threshold = {prob_threshold}")
axes[0].set_xlabel("Predicted Fraud Probability")
axes[0].set_ylabel("Logit-Space Uncertainty (std)")
axes[0].set_title("Decision Protocol Buckets")
axes[0].legend(loc="upper left", fontsize=8, markerscale=3)

# Right: bar chart of bucket composition
bucket_names = ["Auto-approve", "Auto-flag", "Human review"]
bucket_keys = ["auto_approve", "auto_flag", "human_review"]
fraud_counts = [y_val[bucket == b].sum() for b in bucket_keys]
normal_counts = [(bucket == b).sum() - y_val[bucket == b].sum() for b in bucket_keys]
total_counts = [(bucket == b).sum() for b in bucket_keys]

x = np.arange(len(bucket_names))
width = 0.5

bars = axes[1].bar(x, total_counts, width, color=[colors_map[b] for b in bucket_keys], alpha=0.7)
axes[1].bar(x, fraud_counts, width, color="#C44E52", alpha=0.9, label="Fraud")

for i, (total, fraud) in enumerate(zip(total_counts, fraud_counts)):
    pct = total / len(y_val) * 100
    axes[1].text(i, total + 200, f"{total:,}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=9)
    if fraud > 0:
        axes[1].text(i, fraud + 100, f"{fraud} fraud", ha="center", va="bottom", fontsize=8, color="#C44E52")

axes[1].set_xticks(x)
axes[1].set_xticklabels(bucket_names)
axes[1].set_ylabel("Number of Transactions")
axes[1].set_title("Decision Protocol: Bucket Composition")
axes[1].set_yscale("log")
axes[1].legend()

plt.tight_layout()
plt.savefig("../figures/bayesian_decision_protocol.png", dpi=150, bbox_inches="tight")
plt.show()

# print the key operational comparison
print("Decision protocol summary:")
print(f"  Auto-approve: {total_counts[0]:,} transactions ({total_counts[0]/len(y_val)*100:.1f}%) - {fraud_counts[0]} fraud missed")
print(f"  Auto-flag:    {total_counts[1]:,} transactions ({total_counts[1]/len(y_val)*100:.1f}%) - {fraud_counts[1]} fraud caught")
print(f"  Human review: {total_counts[2]:,} transactions ({total_counts[2]/len(y_val)*100:.1f}%) - {fraud_counts[2]} fraud for review")
print(f"\nFraud coverage: {(fraud_counts[1] + fraud_counts[2])}/{y_val.sum()} "
      f"({(fraud_counts[1] + fraud_counts[2])/y_val.sum()*100:.1f}%) caught or sent to review")
print(f"Fraud missed: {fraud_counts[0]}/{y_val.sum()} ({fraud_counts[0]/y_val.sum()*100:.1f}%)")
print(f"Human workload: {total_counts[2]:,} transactions to review ({total_counts[2]/len(y_val)*100:.1f}% of total)")

The decision protocol partitions the 45,569 validation transactions into three buckets. The vast majority (92.7%, or 42,239 transactions) are auto-approved with high confidence: the model predicts low fraud probability and has low uncertainty. Of these, 8 are actual frauds that the protocol misses, a miss rate of 10.1%.

The auto-flag bucket contains 993 transactions (2.2%) where the model is confident they are fraudulent. However, the precision in this bucket is low (0.9%): only 9 of the 993 are actual fraud. The remaining 984 are false positives that would be blocked automatically.

The human review bucket contains 2,337 transactions (5.1%) where the model's uncertainty exceeds the threshold. This is where the Bayesian approach shows its value: 62 of the 79 fraud cases (78.5%) land in this bucket. These are transactions where the model has high epistemic uncertainty, and routing them to human investigators ensures they get proper scrutiny rather than being auto-approved or auto-flagged based on a potentially unreliable prediction.

The combined fraud coverage (auto-flag plus human review) is 89.9%: 71 out of 79 fraud cases are either caught automatically or sent for review. The remaining 8 (10.1%) are missed in the auto-approve bucket. A non-Bayesian model cannot make this distinction. It would either flag all 1,190 high-probability transactions (including the 993 auto-flagged plus some of the review cases) or miss the uncertain ones entirely. The Bayesian protocol reduces the human workload to just 5.1% of transactions while capturing most of the fraud that needs manual attention.

The auto-flag precision is low (0.9%), which reflects the extreme class imbalance and the 0.5 probability threshold. Threshold tuning in Phase 9 will improve this. The key insight here is structural: uncertainty-based routing is a capability that only Bayesian models can provide.

### 5.4 Supervised Model Comparison

Below is a summary comparing the two supervised models. Both use the same linear structure and the same training data. The only difference is that Bayesian LR performs approximate posterior inference over the weights, while standard LR uses a point estimate.

In [ ]:
# Supervised model comparison summary
print("=" * 55)
print("Supervised Model Comparison (Validation Set)")
print("=" * 55)
print(f"\n{'Metric':<25s} {'LR':>12s} {'Bayesian LR':>12s}")
print(f"{'-'*50}")
print(f"{'AUPRC':<25s} {lr_auprc:>12.4f} {blr_auprc:>12.4f}")
print(f"{'ROC-AUC':<25s} {lr_roc_auc:>12.4f} {blr_roc_auc:>12.4f}")
print(f"{'F1 (threshold=0.5)':<25s} {lr_f1:>12.4f} {blr_f1:>12.4f}")

lr_tn, lr_fp, lr_fn, lr_tp = lr_cm.ravel()
blr_tn, blr_fp, blr_fn, blr_tp = blr_cm.ravel()
print(f"{'Precision (t=0.5)':<25s} {lr_tp/(lr_tp+lr_fp):>12.4f} {blr_tp/(blr_tp+blr_fp):>12.4f}")
print(f"{'Recall (t=0.5)':<25s} {lr_tp/(lr_tp+lr_fn):>12.4f} {blr_tp/(blr_tp+blr_fn):>12.4f}")
print(f"\n{'Uncertainty estimates':<25s} {'No':>12s} {'Yes':>12s}")
print(f"{'Decision protocol':<25s} {'No':>12s} {'Yes':>12s}")

print(f"\nDecision protocol (Bayesian LR only):")
print(f"  Auto-approve: {total_counts[0]:,} ({total_counts[0]/len(y_val)*100:.1f}%), {fraud_counts[0]} fraud missed")
print(f"  Auto-flag:    {total_counts[1]:,} ({total_counts[1]/len(y_val)*100:.1f}%), precision={fraud_counts[1]/total_counts[1]:.4f}")
print(f"  Human review: {total_counts[2]:,} ({total_counts[2]/len(y_val)*100:.1f}%), {fraud_counts[2]} fraud for review")
print(f"  Coverage (flag+review): {fraud_counts[1]+fraud_counts[2]}/{y_val.sum()} ({(fraud_counts[1]+fraud_counts[2])/y_val.sum()*100:.1f}%)")

On raw performance metrics, Bayesian LR and standard LR are effectively equivalent. The AUPRC difference (0.6878 vs 0.6825) is marginal and within the range of variation one would expect from minor changes in probability calibration. The ROC-AUC, F1, precision, and recall are nearly identical. This was expected: when two models share the same likelihood and the same training data, the Bayesian treatment primarily adds uncertainty information rather than improving point predictions.

The added value of the Bayesian approach shows up in the decision protocol. Standard LR can only offer a binary choice: flag or approve, based on whether the predicted probability exceeds a threshold. Bayesian LR adds a third option: route uncertain cases to human review. The protocol captured 89.9% of fraud cases (either auto-flagged or sent for review) while requiring human attention on only 5.1% of all transactions. The 8 fraud cases that slipped through the auto-approve bucket (10.1% miss rate) could be reduced by lowering the uncertainty threshold, at the cost of increasing the human review workload.

This three-bucket protocol shows why Bayesian uncertainty is useful in fraud detection, even when it does not improve aggregate metrics. A fraud team with limited investigator capacity can use the uncertainty signal to focus their effort on cases where the model is least certain, rather than reviewing a flood of high-confidence false positives.

## 6. Anomaly Detection Models

The supervised models in Section 5 had access to labeled fraud examples during training. In practice, confirmed fraud labels often arrive with significant delays, sometimes weeks after the transaction occurs (Dal Pozzolo et al., 2017). Anomaly detection offers an alternative framing: learn what normal transactions look like and flag anything that deviates from that pattern.

We train the anomaly detection models on normal transactions only, using the `X_train_normal` subset (181,961 samples). These models never see a single fraud example during training. This makes the comparison with the supervised models informative: the gap between the two groups quantifies how much labeled fraud data is worth, while the anomaly detection performance on its own shows how well fraud can be detected from the structure of normal behavior alone.

We implement two anomaly detection models that mirror the Bayesian vs non-Bayesian comparison from the supervised section: One-Class SVM (non-Bayesian) and a Bayesian Gaussian Mixture Model. An autoencoder is added later to test whether nonlinear representations improve upon these methods.

### 6.1 One-Class SVM

One-Class SVM (Scholkopf et al., 2001) learns a decision boundary that encloses the support of a single-class distribution. The idea is to map the data into a high-dimensional feature space via a kernel function and find the hyperplane that separates the data from the origin with maximum margin. The fraction of training points allowed to fall outside this boundary is controlled by the `nu` parameter, which provides an upper bound on the fraction of outliers.

We use the RBF (radial basis function) kernel, which computes similarity based on Euclidean distance in feature space. This makes the model sensitive to feature scaling, which is why we applied StandardScaler to all 30 features in the preprocessing stage. The standard deviations of V1 through V28 ranged from 0.33 to 1.96 before scaling; without uniform scaling, features with larger variance would dominate the RBF kernel's distance computation.

Training OC-SVM on the full set of 181,961 normal transactions is computationally infeasible because kernel SVM training scales between O(n^2) and O(n^3). We subsample to 15,000 normal transactions, which is large enough to estimate the support of the normal distribution while keeping training time reasonable.

In [ ]:
from sklearn.svm import OneClassSVM
import time

# subsample normal training data for speed (kernel SVM is O(n^2) to O(n^3))
np.random.seed(42)
subsample_idx = np.random.choice(len(X_train_normal), size=15000, replace=False)
X_train_ocsvm = X_train_normal[subsample_idx]
print(f"OC-SVM training set: {X_train_ocsvm.shape[0]} normal transactions (subsampled from {X_train_normal.shape[0]})")

# nu ~ expected fraction of outliers; fraud rate is ~0.17%, so nu=0.01 gives some margin
ocsvm = OneClassSVM(kernel='rbf', nu=0.01, gamma='scale')

print("Training One-Class SVM (this may take a minute)...")
t0 = time.time()
ocsvm.fit(X_train_ocsvm)
elapsed = time.time() - t0
print(f"Training completed in {elapsed:.1f} seconds.")

# score the validation set: decision_function returns signed distance to boundary
# positive = inside boundary (normal), negative = outside (anomalous)
ocsvm_val_scores = ocsvm.decision_function(X_val)
# negate so that higher = more anomalous (for PR curve computation)
ocsvm_val_anomaly_scores = -ocsvm_val_scores

print(f"\nValidation set scored: {len(ocsvm_val_anomaly_scores)} samples")
print(f"Anomaly score range: [{ocsvm_val_anomaly_scores.min():.4f}, {ocsvm_val_anomaly_scores.max():.4f}]")
print(f"Mean anomaly score (normal): {ocsvm_val_anomaly_scores[y_val == 0].mean():.4f}")
print(f"Mean anomaly score (fraud):  {ocsvm_val_anomaly_scores[y_val == 1].mean():.4f}")

The OC-SVM trained on 15,000 subsampled normal transactions in under a second with the `gamma='scale'` setting. The anomaly scores show clear separation between classes: fraud transactions have a mean anomaly score of 0.45, while normal transactions average -0.68. The score range spans from -1.54 (most normal) to 0.77 (most anomalous). The model learned a boundary that separates fraud from normal without ever seeing fraud during training, which confirms that fraud transactions sit in a different region of the feature space.

We set `nu=0.01`, which is above the true fraud rate (0.17%) but gives the boundary some margin. A very tight boundary (`nu` close to 0.002) could underfit rare variations in normal behavior, while a looser boundary increases false positives.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_curve, roc_auc_score

# PR curve and AUPRC
ocsvm_precision, ocsvm_recall, ocsvm_pr_thresholds = precision_recall_curve(y_val, ocsvm_val_anomaly_scores)
ocsvm_auprc = average_precision_score(y_val, ocsvm_val_anomaly_scores)

# ROC curve and ROC-AUC
ocsvm_fpr, ocsvm_tpr, ocsvm_roc_thresholds = roc_curve(y_val, ocsvm_val_anomaly_scores)
ocsvm_roc_auc = roc_auc_score(y_val, ocsvm_val_anomaly_scores)

print(f"OC-SVM Validation Performance:")
print(f"  AUPRC:   {ocsvm_auprc:.4f}")
print(f"  ROC-AUC: {ocsvm_roc_auc:.4f}")

# plot PR and ROC curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PR curve - compare with supervised models
axes[0].plot(ocsvm_recall, ocsvm_precision, color='#2ca02c', lw=2, label=f'OC-SVM (AUPRC={ocsvm_auprc:.4f})')
axes[0].plot(lr_recall, lr_precision, color='#4C72B0', lw=1.5, alpha=0.5, label=f'LR (AUPRC={lr_auprc:.4f})')
axes[0].plot(blr_recall_curve, blr_precision_curve, color='#C44E52', lw=1.5, alpha=0.5, label=f'BLR (AUPRC={blr_auprc:.4f})')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve')
axes[0].legend(loc='upper right', fontsize=9)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1])

# ROC curve
axes[1].plot(ocsvm_fpr, ocsvm_tpr, color='#2ca02c', lw=2, label=f'OC-SVM (AUC={ocsvm_roc_auc:.4f})')
axes[1].plot(lr_fpr, lr_tpr, color='#4C72B0', lw=1.5, alpha=0.5, label=f'LR (AUC={lr_roc_auc:.4f})')
axes[1].plot(blr_fpr, blr_tpr, color='#C44E52', lw=1.5, alpha=0.5, label=f'BLR (AUC={blr_roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.3)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('../figures/ocsvm_pr_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

The OC-SVM achieves an AUPRC of 0.1694 and a ROC-AUC of 0.9276. The ROC-AUC looks reasonable, but the AUPRC is more telling: at 0.17, it is roughly four times lower than the supervised models (LR at 0.68, BLR at 0.69). This gap comes from the information asymmetry between supervised and anomaly detection: OC-SVM has never seen a fraud example, so it can only detect fraud to the extent that fraudulent transactions deviate from the learned normal boundary.

The PR curve shows that OC-SVM can achieve moderate recall (up to roughly 0.8) but only at very low precision. This is typical for anomaly detectors on heavily imbalanced data: many normal transactions also fall near or outside the decision boundary, producing a large number of false positives. The ROC-AUC stays high (0.93) because the false positive rate is measured against the large pool of ~45,000 negatives, which dilutes the impact of false positives.

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score

# use the OC-SVM's native predict: +1 = normal, -1 = anomaly
ocsvm_val_preds_raw = ocsvm.predict(X_val)
# convert to our label convention: 0 = normal, 1 = fraud
ocsvm_val_preds = (ocsvm_val_preds_raw == -1).astype(int)

ocsvm_cm = confusion_matrix(y_val, ocsvm_val_preds)
ocsvm_f1 = f1_score(y_val, ocsvm_val_preds)
tn, fp, fn, tp = ocsvm_cm.ravel()

print(f"OC-SVM Confusion Matrix (using native predict boundary):")
print(f"  TP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}")
print(f"  Precision: {tp/(tp+fp):.4f}")
print(f"  Recall:    {tp/(tp+fn):.4f}")
print(f"  F1:        {ocsvm_f1:.4f}")
print(f"  False alarm rate: {fp/(fp+tp):.1%}")

# plot confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(ocsvm_cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Normal', 'Fraud'], yticklabels=['Normal', 'Fraud'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('OC-SVM Confusion Matrix (Validation)')
plt.tight_layout()
plt.savefig('../figures/confusion_matrix_ocsvm.png', dpi=150, bbox_inches='tight')
plt.show()

At the native decision boundary, OC-SVM detects 67 of 79 fraud transactions (recall = 0.848), but generates 1,611 false positives, yielding a precision of just 4.0% and an F1 of 0.076. The false alarm rate is 96.0%: out of every 100 transactions flagged as anomalous, only 4 are actual fraud. This is worse than LR's 94% false alarm rate at the default 0.5 threshold, and LR had the advantage of labeled fraud during training.

The high recall shows that OC-SVM does catch most fraud. The poor precision is the main challenge for anomaly detection on this dataset: the decision boundary must be tight enough to exclude fraud but loose enough to include the full variety of normal transactions. With `nu=0.01`, the model allows roughly 1% of training points to fall outside the boundary, which translates to hundreds of normal validation transactions being flagged alongside the true anomalies. Threshold tuning in Phase 9 will select a better operating point on the PR curve.

In [ ]:
# score distribution: how well does OC-SVM separate fraud from normal?
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.hist(ocsvm_val_anomaly_scores[y_val == 0], bins=100, alpha=0.6, density=True,
        color='#4C72B0', label='Normal')
ax.hist(ocsvm_val_anomaly_scores[y_val == 1], bins=30, alpha=0.7, density=True,
        color='#C44E52', label='Fraud')

# mark the native decision boundary (score = 0 after negation)
ax.axvline(x=0, color='black', linestyle='--', lw=1.5, alpha=0.7, label='Decision boundary')

ax.set_xlabel('Anomaly Score (negated decision function)')
ax.set_ylabel('Density')
ax.set_title('OC-SVM Score Distribution: Normal vs Fraud')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/ocsvm_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# quantify overlap
fraud_above_boundary = (ocsvm_val_anomaly_scores[y_val == 1] > 0).mean()
normal_above_boundary = (ocsvm_val_anomaly_scores[y_val == 0] > 0).mean()
print(f"Fraction of fraud above decision boundary (score > 0): {fraud_above_boundary:.1%}")
print(f"Fraction of normal above decision boundary (score > 0): {normal_above_boundary:.2%}")

The score distribution plot shows clear but imperfect separation. 84.8% of fraud transactions fall above the decision boundary (anomaly score > 0), compared to 3.54% of normal transactions. The bulk of normal transactions cluster tightly at negative anomaly scores (well inside the learned boundary), while fraud transactions are spread across a wider range of positive scores. There is some overlap in the region around the boundary, which is the source of both the false positives and the missed frauds.

The 3.54% of normal transactions flagged as anomalous corresponds to roughly 1,611 false positives in the validation set (3.54% of 45,490 negatives), consistent with the confusion matrix above. This false positive rate is partly a consequence of setting `nu=0.01` during training, which allowed approximately 1% of training normals to fall outside the boundary. The validation set sees a slightly higher rate (3.5%) because validation data can include normal patterns not represented in the 15,000-sample training subsample.

OC-SVM detects most fraud without ever seeing a fraud example, though at a substantial precision cost compared to the supervised models. The AUPRC of 0.1694 will serve as our non-Bayesian anomaly detection baseline for comparison with the Bayesian GMM in the next section.